In [2]:
import numpy as np
import pandas as pd
import cv2 as cv
import ffmpeg

import torch
import torch.nn as nn
import torchvision.models as models
import pytorch_lightning as pl

from ncps.torch import CfCCell

In [3]:
batch_size = 16 # placeholder well decide the actual value later
image_h=256
image_w=384
S=8
image_feature_size=960
hidden_size=512
backbone_units=1024
ts = torch.ones(batch_size)

In [4]:
class CoordConv(nn.Module):
   
    def __init__(self, in_channels, out_channels, kernel_size=1, padding=0):
        super().__init__()

        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, padding=padding)

    def forward(self, x):
        batch_size, _, h, w = x.size()
        
    
        y_grid = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(batch_size, 1, h, w)
        x_grid = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(batch_size, 1, h, w)
        
        
        x_coord = torch.cat([x, y_grid, x_grid], dim=1)
        return self.conv(x_coord)

In [5]:
class GazeLLNArch(pl.LightningModule):

    def __init__(self, image_h: int = 256, image_w: int = 384, S: int = 8, 
                 hidden_size: int = 512, backbone_units: int = 1024):
        super().__init__()
        
        self.save_hyperparameters()

        
        self.hmap_h = self.hparams.image_h // self.hparams.S
        self.hmap_w = self.hparams.image_w // self.hparams.S
        self.hmap_feature_size = self.hmap_h * self.hmap_w

        
        mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        self.feature_extractor = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.image_feature_size = 960 

        
        self.coordconv = CoordConv(in_channels=1, out_channels=1, kernel_size=3, padding=1)

        
        cfc_input_size = self.image_feature_size + self.hmap_feature_size
        self.cfc_cell = CfCCell(
            input_size=cfc_input_size,
            hidden_size=self.hparams.hidden_size,
            backbone_activation="lecun_tanh",
            backbone_units=self.hparams.backbone_units,
            backbone_layers=1
        )
        
        
        self.project = nn.Linear(self.hparams.hidden_size, self.hmap_feature_size)

    def forward(self, image, prev_hmap, hx, ts):
        """
        Args:
            image: Visual stimulus tensor of shape (B, 3, H, W)
            prev_hmap: Previous fixation heatmap of shape (B, hmap_h, hmap_w) or (B, 1, hmap_h, hmap_w)
            hx: Hidden state tensor for the CfCCell
            ts: Elapsed time timespan tensor of shape (B,)
        """
        
        vis_features = self.feature_extractor(image)
        vis_features = self.pool(vis_features).flatten(1)  # Shape: (B, 960)

        # Ensure prev_hmap has a channel dimension (B, 1, hmap_h, hmap_w)
        if prev_hmap.dim() == 3:
            prev_hmap = prev_hmap.unsqueeze(1)
            
    
        # hmap_coords = self.coordconv(prev_hmap).flatten(1)
        hmap_coords = prev_hmap.flatten(1) # Shape: (B, hmap_h * hmap_w) (using coordconv before flattenning might be useless)
        
        
        x = torch.cat([vis_features, hmap_coords], dim=1)
        x, hx = self.cfc_cell(x, hx, ts)
        
        
        out_hmap = self.project(x)

        # Implementing Softmax?
        out_hmap = out_hmap.view(-1, self.hmap_h, self.hmap_w)
        out_hmap = torch.softmax(out_hmap.flatten(1), dim=1).view(-1, self.hmap_h, self.hmap_w)

        return out_hmap, hx

In [6]:
# Sample Forward Pass

# Instantiating the model
model = GazeLLNArch(
    image_h=256,
    image_w=384,
    S=8,
    hidden_size=512,
    backbone_units=1024
)
model.eval()  # puts model in inference mode, disables dropout etc.

# Creating dummy inputs matching expected shapes
B = 16  # using smaller batch size

dummy_image   = torch.randn(B, 3, 256, 384)        # (B, C, H, W)
dummy_hmap    = torch.zeros(B, 256//8, 384//8)     # (B, hmap_h, hmap_w) = (B, 32, 48)
dummy_hx      = torch.zeros(B, 512)                # (B, hidden_size)
dummy_ts      = torch.ones(B, 1)                   # (B,) elapsed time, all 1s for now

# Running one forward pass
with torch.no_grad():  # don't compute gradients during testing
    out_hmap, new_hx = model(dummy_image, dummy_hmap, dummy_hx, dummy_ts)

# Checking output shapes
print("Output heatmap shape:", out_hmap.shape)   # should be (16, 32, 48)
print("New hidden state shape:", new_hx.shape)   # should be (16, 512)

Output heatmap shape: torch.Size([16, 32, 48])
New hidden state shape: torch.Size([16, 512])


In [14]:
def Get_frame(vid):
    curr_ret, curr_frame = vid.read()

    if curr_ret == False:
        return curr_ret, curr_frame
    
    curr_frame = cv.cvtColor(curr_frame, cv.COLOR_BGR2RGB)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    curr_frame = torch.tensor(curr_frame).permute(2, 0, 1).float()              # (H,W,C) -> (C,H,W)

    # Image Normalization
    curr_frame = curr_frame / 255.0                                             # scale to [0, 1]
    curr_frame = (curr_frame - mean) / std
    
    curr_frame = curr_frame.unsqueeze(0)                                        # (C,H,W) -> (1,C,H,W)

    return curr_ret, curr_frame

def Preprocess_video():
    (ffmpeg
       .input("sample_video.mp4")
       .filter('fps', fps = 24, round = 'up')
       .filter('scale', 384, 256)
       .output("24_fps_video.mp4",
               vcodec='libx264rgb',
               pix_fmt='rgb24'
            )
       .overwrite_output()
       .run()
    )


In [15]:
Preprocess_video()
capt = cv.VideoCapture("24_fps_video.mp4")
frames_list = []
while True:
    ret, frame = Get_frame(capt)

    if ret == False:
        break

    frames_list.append(frame)

frames = np.array(frames_list)           # Shape: (number of frames, Batch size, Channel, Height, Width)